# Late Interaction and Learned Sparse Retrieval: ColBERT and SPLADE

`embedding-dimension-lower-bounds` proved a single pooled vector has a **sign-rank ceiling** —
relevance patterns it cannot realize below a critical dimension. This notebook walks the two
architectures that **escape** that ceiling, by different mechanisms: **ColBERT** keeps one vector per
*token* and scores by MaxSim; **SPLADE** expands into a high-dimensional *sparse* lexical space.

Every claim is checked by an `assert` in the companion module `late_interaction_learned_sparse.py`,
which **owns every number** the topic page and lab display. It imports DPR (the dot-product anchor),
the embedding-dimension topic (the wall + the qrel loss), and BM25 (the lexical world) — and never
reimplements them. The one *provable* fact amid an empirical escape: MaxSim reduces exactly to the
DPR dot product when there is one vector per item.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks" / "late-interaction-learned-sparse"))
import numpy as np
import late_interaction_learned_sparse as li

# MaxSim of one query (3 token vectors) against one document (4 token vectors):
Q = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]])
D = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]])
print("MaxSim score:", li.maxsim_score(Q, D), " (each query token max-matches a document token)")

## Movement 1 — MaxSim, and the m=1 collapse

Late interaction scores a query against a document by
$$S(q, d) = \sum_{i \in q} \max_{j \in d} \langle E(q_i), E(d_j)\rangle.$$
This is a **max of inner products** — piecewise-linear, not a single bilinear form $q^\top W d$ — so
the rank-$\le d$ argument that bounds a single pooled vector does not apply. The one thing we can prove
cleanly is the boundary case: with **one vector per item** the max is over a singleton, so MaxSim is
*exactly* the dual-encoder dot product imported from DPR. That is the anchor; the escape itself is
demonstrated, not proved.

In [ ]:
li.test_maxsim_reduces_to_dot_at_m1()
li.test_maxsim_is_max_of_linear()

## Movement 2 — multi-vector escapes the single-vector wall

Reusing the all-pairs qrel from the embedding-dimension topic, we compare a single-vector model and a
MaxSim model with two vectors per document — **the same free-embedding optimizer**, so the gap is
purely the multi-vector effect (one cloud, not a comparison across optimizers). Past the single-vector
critical $n$, the single vector's row-order accuracy collapses while MaxSim stays perfect. The escape
is **empirical**: no multi-vector sign-rank lower bound is known (the LIMIT paper explicitly defers
it). The cost is storage — one vector per token.

In [ ]:
li.test_multivector_escapes_wall()
li.test_maxsim_critical_n_exceeds_single()
li.test_storage_cost()

print("\nsingle-vector wall vs MaxSim escape (row-order accuracy by corpus size):")
for row in li.maxsim_escape_curve(li.ESCAPE_D, li.ESCAPE_N_GRID, li.N_DOC_VECS):
    print(f"  n = {row['n']:>2}:  single = {row['single']:.3f},  MaxSim(m={li.N_DOC_VECS}) = {row['maxsim']:.3f}")

## Movement 3 — SPLADE, the lexical escape

The other escape is dimensional: expand into the high-dimensional **sparse** vocabulary space
($|V| \gg d$), living in BM25's inverted index. A term's weight is
$w_j = \max_{i} \log(1 + \mathrm{ReLU}(\ell_{ij}))$ over the input tokens, and a **FLOPS regularizer**
$\mathcal{L} = \sum_j \bar a_j^2$ controls sparsity. Learned expansion fixes the vocabulary mismatch
BM25 cannot: a query whose terms are *absent* from a relevant document still retrieves it. (The
"MLM logits" here are a synthetic association stand-in, not a trained BERT.)

In [ ]:
li.test_splade_fixes_vocab_mismatch()
li.test_flops_controls_sparsity()

vocab = li._vocab_of(li.BM25_CORPUS)
qw = li.splade_weights(li.SPLADE_QUERY, vocab)
inv = {j: t for t, j in vocab.items()}
active = sorted([(inv[j], round(float(qw[j]), 3)) for j in np.flatnonzero(qw)], key=lambda kv: -kv[1])
print(f"\nSPLADE weights for the mismatch query {li.SPLADE_QUERY!r} (all learned expansion):")
print(" ", active)

## Finance case study and the viz constants

The two escapes map to the two finance failure modes of the single vector: combinatorial multi-company
queries (ColBERT MaxSim) and conjunctive lexical mismatch (SPLADE expansion). The constants below are
mirrored to the decimal in `LateInteractionLaboratory.tsx`; the TypeScript recomputes only closed
forms (L0 counts, MaxSim of the baked token vectors).

In [ ]:
_ = li.finance_demo()
print()
li.viz_constants()